# Decision Trees: Understanding Feature Importance

## Introduction

In Lesson 2, we built a Logistic Regression classifier and evaluated it with accuracy, precision, recall, and ROC/AUC. Today we explore a fundamentally different model: **Decision Trees**. Instead of a linear boundary (like LR's diagonal line in feature space), decision trees make predictions by asking a series of yes/no questions — creating a tree-like structure of decisions.

> ❓ **A key question motivating this lesson:** logistic regression can only draw straight-line boundaries between classes. But what if the real pattern in the Nepal data is non-linear? What if buildings with *both* old age AND weak foundations are disproportionately at risk — a combination that a single linear term cannot capture? Decision Trees are designed to handle exactly this kind of interaction.

**The Nepal Gorkha context matters here.** After the April 2015 magnitude-7.8 earthquake, structural engineers documented that damage patterns were highly *interactive*: a mud-mortar stone building that was also very tall and old suffered catastrophically, while a short mud-mortar building often survived. This interaction — age AND foundation AND height together — is exactly what decision trees can model naturally by asking sequential questions about combinations of features, rather than treating each feature independently.

**Why we study both LR and Decision Trees:**

| | Logistic Regression | Decision Tree |
|---|---|---|
| **Boundary type** | Diagonal line (linear) | Axis-aligned rectangles |
| **Feature interactions** | No (additive model) | Yes (sequential splits) |
| **Interpretability** | Odds ratios (global) | Tree path (local, case-by-case) |
| **Best when** | Additive effects dominate | Non-linear interactions matter |

Together, LR and Decision Trees form the core of classical binary classification — two complementary tools you will use throughout Unit 2.

By the end of this lesson, you will be able to:

1. Explain how **Decision Trees** make predictions through recursive splitting and how their decision boundaries differ from logistic regression
2. Define **Gini impurity** and describe how trees use it to find the best feature splits
3. Describe the **bias-variance tradeoff** and demonstrate how `max_depth` controls it
4. Implement a **three-way train/validation/test split** for honest hyperparameter tuning
5. Use a **validation curve** to find the optimal `max_depth` for the Nepal dataset
6. Extract and interpret **feature importance** scores from a fitted Decision Tree
7. Visualize the tree structure with `plot_tree` and trace a prediction path from root to leaf
8. Compare Decision Tree performance to Logistic Regression on the same Nepal data


## Part 1: How Decision Trees Work — Recursive Splitting

Decision trees make predictions by asking a series of **yes/no questions** about feature values. Each question splits the data into two groups, and the process repeats until we reach a final prediction (a leaf node).

### The Core Idea

Think of a decision tree like playing "20 Questions" about a building:

```
Is the building's foundation strong?
    ↓
  NO: Ask another question...
    ↓
Is the building very tall (height > 15 ft)?
    ↓
  YES: Predict "Not Severely Damaged"
  NO:  Predict "Severely Damaged"
```

Each question **partitions the data** into two groups. This is called **recursive splitting** because the exact same process — find the best question, split the data — repeats at every node.

### A Simple 3-Level Tree Example

```
                     Is foundation_type ≤ 2?
                          /      \
                       YES        NO
                      /            \
                Is height > 15?    Predict: SEVERE
                /        \
              YES        NO
              /            \
        Predict:       Predict:
      NOT SEVERE        SEVERE
```

**Reading this tree:**
- **Root node (top):** First question: "Is foundation_type ≤ 2?" (where 2 = ordinal-encoded weak foundation)
  - If YES → Go left, ask another question
  - If NO → Go right, predict SEVERE immediately (strong foundations tend to survive)
- **Interior nodes (middle):** More questions narrow down the groups further
- **Leaf nodes (bottom):** Final predictions — these are the answer

**What this tree learns:**
- Weak foundation + tall building → Severely damaged
- Weak foundation + short building → Not severely damaged
- Strong foundation → Always severely damaged (possibly RC, which ironically may be in denser urban construction)


### Why "Recursive"?

**Recursive** means repeating the same process at each step. At every node, the tree:

1. Examines all remaining data in that subtree
2. Tries every possible feature and every possible threshold
3. Finds the one split that most reduces **Gini impurity** (covered in Part 5)
4. Divides data into two groups
5. **Repeats** on each group independently until a stopping condition is met

The stopping conditions are:
- Node is **pure** (Gini = 0, all one class) — no further splitting can improve purity
- Node has too few samples (controlled by `min_samples_split`) — prevents splitting on tiny subgroups
- Tree has reached maximum depth (`max_depth` parameter) — the primary regularization control

**Applied to Nepal data:** a tree might learn:
```
Is foundation_type in {mud-mortar, stone}?
  YES → Is age_building > 25 years?
    YES → Is height_ft_pre_eq > 15?
      YES → 89% of these buildings: Severe (leaf)
      NO  → 63% of these buildings: Severe (leaf)
    NO  → 45% of these buildings: Not Severe (leaf)
  NO → 18% of these buildings: Severe (leaf)
```

This tree asks about **combinations** of features — something logistic regression models with additive terms (`β₁·age + β₂·foundation`) cannot fully capture.

> 💡 **Unlike logistic regression** (which defines one global boundary), a decision tree defines many *local* boundaries — each node is a separate boundary that only applies to the subset of data that reached that node. This is what makes trees capable of capturing complex, non-linear patterns.

> ⚠️ **The flexibility is a double-edged sword.** With no depth limit, a tree will create separate rules for every individual training example — achieving 100% training accuracy but predicting nonsense on new data (overfitting). `max_depth` is the critical regularization parameter that prevents this.


## Part 2: Visualizing Tree Splits in Feature Space

To build intuition, imagine a 2D world with only two features: `age_building` and `height_ft_pre_eq`.

### Decision Tree Boundaries: Axis-Aligned Rectangles

A decision tree creates **axis-aligned rectangular splits** — each split is a vertical or horizontal line:

```
height_ft │
      20  │ [Not Severe] │ [Severe]     │
      15  │──────────────┼──────────────┤ ← "height ≤ 15?" split
      10  │ [Severe]     │ [Not Severe] │
       5  │──────────────┼──────────────┤
       0  └────────────────────────────────> age_building
          0              10             20
                         ↑
                   "age ≤ 10?" split
```

Each split corresponds to one decision node in the tree:
- `age ≤ 10?` → vertical line at age = 10
- `height ≤ 15?` → horizontal line at height = 15

The result: a **checkerboard of rectangles**, each assigned a predicted class.

> 📌 **Why "axis-aligned"?** Each split tests only one feature at a time (`feature X ≤ threshold`). The split line is always perpendicular to one axis. This is the fundamental constraint of decision trees — and why they struggle with diagonal boundaries (which require combining two features simultaneously).

### Logistic Regression Boundary: Linear (Diagonal)

By contrast, logistic regression creates a **single straight diagonal boundary**:

```
height_ft │
      20  │              ╱ [Severe]
      15  │           ╱
      10  │        ╱  [Not Severe]
       5  │     ╱
       0  │  ╱
          └────────────────────────> age_building
```

The LR boundary is determined by `z = β₀ + β₁·age + β₂·height = 0` — a single equation involving both features simultaneously.

### Side-by-Side Comparison

| | Decision Tree | Logistic Regression |
|---|---|---|
| **Boundary shape** | Axis-aligned rectangles | Single diagonal line |
| **Handles checkerboard** | ✓ Naturally | ✗ Struggles |
| **Handles diagonal** | ✗ Requires many splits | ✓ Naturally |
| **Complexity** | Grows with depth | Fixed (one boundary) |

> 🧠 **When is each better?** If the true pattern is "buildings where BOTH age > 20 AND height > 15 are at risk," a tree handles this with two axis-aligned cuts. If the true pattern is "older AND taller buildings proportionally increase risk," the LR diagonal captures it. In practice, you run both and compare.


## Part 3: Underfitting vs Overfitting — The max_depth Spectrum

`max_depth` is a **hyperparameter** that limits how deep the tree can grow. It directly controls whether the tree underfits or overfits.

### Shallow Tree (max_depth = 1 or 2): Underfitting

```
height_ft │
       20 │ [Not Severe]  │  [Severe]
       15 │───────────────┤
       10 │               │
        5 │               │
        0 └─────────────────────────> age_building
                          ↑
               Only one split (max_depth=1)
```

**What happens:**
- Only 1 or 2 splits total — the tree asks one or two questions
- The boundary is oversimplified compared to the true pattern
- The model has **high bias** (wrong assumptions about the data)

| Metric | Typical value |
|--------|--------------|
| Training accuracy | ~65% |
| Validation accuracy | ~65% |
| Gap (overfit?) | ~0% (no overfitting — too simple to overfit!) |
| **Diagnosis** | UNDERFITTING |

> 💡 **Underfitting** means the model is too simple to capture the true patterns in the data. Both training and test accuracy are poor. The fix is more model complexity (deeper tree).


### Medium Tree (max_depth ≈ 6): The Sweet Spot

```
height_ft │
       20 │ A │ B │ C │ D  ← Multiple regions
       15 │───┼───┼───┼───
       10 │ E │ F │ G │ H
        5 │───┼───┼───┼───
        0 └─────────────────> age_building
```

**What happens:**
- Many splits create fine-grained regions
- Captures the true non-linear patterns in the data
- Has moderate complexity — not too simple, not too complex

| Metric | Typical value |
|--------|--------------|
| Training accuracy | ~74% |
| Validation accuracy | ~72% |
| Gap | ~2% (healthy, expected) |
| **Diagnosis** | GOOD GENERALIZATION |

### Deep Tree (max_depth = 20+): Overfitting

```
height_ft │
       20 │╱╲╱╲╱╲╱╲╱╲  ← Jagged, complex boundary
       15 │╲╱╲╱╲╱╲╱╲╱
       10 │╱╲╱╲╱╲╱╲╱╲
        5 │╲╱╲╱╲╱╲╱╲╱
        0 └────────────> age_building
```

**What happens:**
- Hundreds or thousands of leaf nodes
- Memorizes training data — creates separate rules for individual buildings
- The jagged boundary follows noise, not true patterns

| Metric | Typical value |
|--------|--------------|
| Training accuracy | ~99% |
| Validation accuracy | ~68% |
| Gap | ~31% (severe overfitting) |
| **Diagnosis** | OVERFITTING |

### Visual Summary: The Validation Curve

```
Accuracy
  99%  ┤                    Training ───────────────────
  90%  ┤                  ╱
  80%  ┤              ╱╱
  72%  ┤         ╱╱╱╱  ─ ─ Validation ─ ─╮
  68%  ┤  ─ ─ ─              ─ ─ ─ ─ ─ ─  ╲ ─ ─ ─ ─ ─
  65%  ┤ ╱ (start)
       │
       │←Underfit→│←Optimal→│←────── Overfit ────────→
       └──────────┬──────────┬──────────────────────────> max_depth
                  2          6                        20
```

The **optimal max_depth** is where the validation curve peaks. Beyond that point, training accuracy keeps rising but validation accuracy falls — classic overfitting.

> 📌 **Why the validation curve eventually *decreases*:** at very high max_depth, the tree memorizes training noise so specifically that it misclassifies unseen data. The model's "knowledge" is too specific to the training buildings to generalize.


## Part 4: Bias-Variance Tradeoff and Hyperparameters

The max_depth spectrum we just saw is a specific instance of a fundamental principle: the **bias-variance tradeoff**.

### Bias: The Error from Oversimplification

**Bias** measures how much the model's predictions systematically miss the true pattern.

- **High bias**: the model's assumptions are too restrictive — it can't capture the true relationship even with unlimited data
- **Low bias**: the model is flexible enough to fit complex patterns
- Example: a tree with `max_depth=1` has high bias — it assumes the entire world can be divided by a single feature threshold

**Symptom:** both training accuracy AND test accuracy are poor. The model fails even on data it has seen.

### Variance: The Error from Oversensitivity to Noise

**Variance** measures how much the model's predictions change when trained on a different sample of data.

- **High variance**: small changes in training data produce large changes in predictions — the model is fitting noise
- **Low variance**: predictions are stable across different training samples
- Example: a tree with `max_depth=20` has high variance — it memorizes every training building, so different training sets produce completely different trees

**Symptom:** training accuracy is high but test accuracy is much lower. The model works on what it has seen but fails on new data.

### The Fundamental Tradeoff

```
Total Error = Bias² + Variance + Irreducible Noise

Error
  │  Bias² (↓ as depth ↑)
  │──────╲
  │       ╲                    Variance (↑ as depth ↑)
  │        ╲                  ╱────────────
  │         ╲                ╱
  │          ╲              ╱  Total Error
  │           ╲────────────╱
  │                  ↑
  │            Optimal depth
  └─────────────────────────────────> max_depth
```

At the **optimal depth**, the sum of bias and variance is minimized — this is the depth that produces the best generalization.

> 🧠 **The bias-variance tradeoff is universal.** It applies to every model type, not just decision trees. In logistic regression, regularization (`C` parameter) controls the same tradeoff. In neural networks, it's the number of layers and neurons. The specific control knob changes; the underlying principle is always the same.

### max_depth Summary

| max_depth | Bias | Variance | Behavior |
|-----------|------|----------|----------|
| 1-3 | High | Low | Underfitting — too simple |
| 5-8 | Medium | Medium | Good generalization — the sweet spot |
| 15+ | Low | High | Overfitting — memorizes training data |
| None (unlimited) | Very low | Very high | Extreme overfitting |

### The Nepal Validation Curve (Example Expected Output)

When you run the hyperparameter loop on Gorkha data, expect something like:

| max_depth | Training Accuracy | Validation Accuracy | Interpretation |
|-----------|------------------|--------------------|--------------
| 1 | 64.8% | 64.6% | Barely better than majority baseline (underfitting) |
| 3 | 70.1% | 69.8% | Capturing main patterns, small train/val gap |
| 6 | 74.2% | 71.9% | Good generalization, modest gap |
| 8 | 80.5% | 71.5% | Slight overfitting beginning |
| 12 | 93.2% | 69.8% | Clear overfitting — large gap |
| 20 | 99.1% | 67.4% | Severe overfitting — memorizing individual buildings |

> 📌 **The key pattern:** validation accuracy peaks around depth 5-7 and then declines, while training accuracy keeps climbing. The gap between the two curves widens as depth increases — this widening gap is the visual signature of overfitting.


## Part 5: Gini Impurity — How Trees Find the Best Split

How does a decision tree decide *which feature* to split on and *at what threshold*? It uses **Gini impurity** to measure the quality of each potential split.

### Intuition: Pure vs Mixed Groups

Imagine sorting buildings into two buckets based on a rule:

```
Rule: "foundation_type ≤ 2"

Bucket 1 (YES — weak foundations, 350 buildings):
  300 severe, 50 not severe
  → Mostly one class → RELATIVELY PURE

Bucket 2 (NO — strong foundations, 150 buildings):
  0 severe, 150 not severe
  → All one class → COMPLETELY PURE (Gini = 0!)
```

A **pure bucket** (all one class) gives confident predictions. A **mixed bucket** is uncertain.

### Gini Formula

For a node containing two classes with proportions `p` (severe) and `1-p` (not severe):

```
Gini = 2 × p × (1 - p)
```

| p (fraction severe) | Gini | Interpretation |
|--------------------|----|----------------|
| 0.0 (all not severe) | 0.00 | Pure node — confident prediction: NOT severe |
| 0.1 | 0.18 | Mostly not severe |
| 0.5 (half/half) | 0.50 | Maximally uncertain |
| 0.9 | 0.18 | Mostly severe |
| 1.0 (all severe) | 0.00 | Pure node — confident prediction: SEVERE |

> 📌 Gini is 0 at the extremes (pure) and peaks at 0.5 (maximally mixed). The tree algorithm seeks to minimize Gini — it wants pure buckets.

### How Trees Use Gini to Split

At each node, the tree evaluates every possible split and picks the one that most reduces total Gini impurity. Example:

```
Before split (500 buildings, 60% severe):
  p = 0.6, Gini = 2 × 0.6 × 0.4 = 0.48

Candidate split: "foundation_type ≤ 2?"
  Left (350 buildings, 86% severe):  p=0.86, Gini = 2×0.86×0.14 = 0.24
  Right (150 buildings, 0% severe):  p=0.00, Gini = 0.00

Weighted Gini after split = (350/500)×0.24 + (150/500)×0.00 = 0.168

Gini reduction = 0.48 - 0.168 = 0.312  ← A large reduction → this is a GOOD split
```

The tree compares this Gini reduction against all other possible splits and picks the one with the highest reduction.

> 🧠 **Why Gini instead of accuracy?** Gini is differentiable and can be computed efficiently at each node. More importantly, it measures node purity — not just whether the majority class changed. Two splits can have the same majority-class accuracy but very different Gini values; the lower Gini split produces more confident leaf predictions.


### Why Decision Trees Use OrdinalEncoder (not OHE)

In Lesson 2, we used `OneHotEncoder` for logistic regression. For decision trees, we use `OrdinalEncoder` instead. Why?

**Logistic regression requires OHE** because:
- LR computes `z = β₀ + β₁·x₁ + ...` — a weighted sum
- With ordinal integers (0, 1, 2 for foundation types), LR would treat 2 as "twice as much" as 1
- OHE gives each category its own independent coefficient

**Decision trees work with OrdinalEncoder because:**
- Trees split on thresholds like `foundation_type ≤ 2.5`
- They don't compute weighted sums — they just compare a value to a threshold
- The ordering imposed by ordinal integers doesn't matter: `foundation_type ≤ 2.5` correctly separates encoded category 0, 1, 2 from category 3, 4 regardless of what order the categories were assigned
- OHE would create many more columns, slowing down the split search

```python
# OHE for LR:
foundation_type → 5 binary columns (foundation_RC, foundation_Mud, ...)
# OrdinalEncoder for DT:
foundation_type → 1 integer column (0, 1, 2, 3, 4)
```

> 📌 **In practice, sklearn's `DecisionTreeClassifier` can also handle ordinal-encoded categorical data correctly by searching for the optimal split point across all possible threshold values.** It will find the split `foundation_type ≤ 0.5` (separating category 0 from categories 1-4) if that's the best split, even though the ordinal ordering doesn't imply anything meaningful about the categories.


## Part 6: Decision Trees vs Logistic Regression — A Detailed Comparison

We have now built both models on the same Nepal data. Let's compare them across the dimensions that matter most for data science practice.

### Boundary Shape and Flexibility

| | Decision Tree | Logistic Regression |
|---|---|---|
| **Boundary type** | Axis-aligned rectangles | Linear hyperplane |
| **Can capture** | Checkerboard, step-function patterns | Linear/diagonal patterns |
| **Strength** | Non-linear interactions (age AND foundation) | Additive effects (each feature independently) |
| **Weakness** | Diagonal patterns need many splits | Non-linear interactions need feature engineering |

### Interpretability

**Logistic Regression:** interpretable via **odds ratios**
```
"RC foundation → odds ratio = 0.2 → having RC foundation reduces
 the odds of severe damage by 80%"
```
This is a global statement about the average effect across all buildings.

**Decision Trees:** interpretable via **tree path tracing**
```
"For this specific building: foundation_type is strong (>2.5) →
 AND height < 15 → Predict: NOT severe"
```
This is a local explanation for a specific prediction — very intuitive for non-technical stakeholders.

### Feature Importance

| | Logistic Regression | Decision Tree |
|---|---|---|
| **Metric** | Odds ratio (exp of coefficient) | Gini reduction (summed across all splits) |
| **Shows direction** | ✓ Yes (>1 = increases risk, <1 = decreases) | ✗ No (only magnitude) |
| **Shows magnitude** | ✓ Yes | ✓ Yes |
| **Accounts for interactions** | ✗ No (additive model) | ✓ Yes (split sequences capture interactions) |

### Comprehensive Comparison

| Aspect | Logistic Regression | Decision Tree |
|--------|---------------------|---------------|
| **Boundary shape** | Linear (diagonal) | Rectangular (axis-aligned) |
| **Handles nonlinearity** | No (requires feature engineering) | Yes (naturally) |
| **Interpretability** | Coefficients/odds ratios | Visual tree + feature importance |
| **Overfitting risk** | Low (linear) | High (needs max_depth control) |
| **Encoding required** | OneHotEncoder (nominal) | OrdinalEncoder (any type) |
| **Hyperparameters** | `C` (regularization), `max_iter` | `max_depth`, `min_samples_split` |
| **Training speed** | Fast (convergence usually <1000 iter) | Fast (O(n·features·log n)) |
| **Probability output** | Calibrated probabilities | Less calibrated (uses leaf proportions) |
| **Class imbalance** | Needs `class_weight='balanced'` | Needs `class_weight='balanced'` |

> 🧠 **Neither model is universally better.** Logistic regression is better when the true relationship is additive and linear (each feature contributes independently). Decision trees are better when features interact non-linearly. In practice, you run both and compare validation metrics — exactly what we do in this lesson.


### The Class Imbalance Problem

What happens when one class is much rarer than the other?

**Example:** 1% of buildings have a rare foundation type. A model that ignores this type and predicts "other" for every building gets 99% accuracy — but it captures 0% of that rare foundation type correctly.

**In our Nepal dataset:** 64% severe vs 36% not severe — a mild imbalance. Both models handle this acceptably by default. But in more extreme cases (e.g., 95% vs 5%), both logistic regression and decision trees default to predicting the majority class too often.

**Fixes (same for both models):**
1. `class_weight='balanced'`: automatically weights minority class higher in the loss function
2. **Threshold adjustment**: lower the decision threshold from 0.5 to 0.3 to flag more minority-class cases
3. **Oversampling** (SMOTE): synthetically generate minority-class training examples

> 💡 **For disaster response:** we should always consider lowering the threshold. Missing a severely damaged building (false negative) is far more costly than a false alarm. A threshold of 0.4 instead of 0.5 will increase recall at the cost of some precision — a reasonable trade-off.

### Summary: Which Model for Which Job?

| Task | Recommendation | Reason |
|------|---------------|--------|
| Understanding *why* a building was classified | Decision Tree | Visual path tracing from root to leaf |
| Reporting probabilities to engineers | Logistic Regression | Calibrated probabilities are more reliable |
| Handling interactions (age AND foundation) | Decision Tree | Splits capture combinations naturally |
| Maximizing recall (catch every damaged building) | Either + threshold adjustment | Threshold tuning applies to both |
| Communicating to policy makers | Decision Tree (shallow) | "If foundation is weak AND building is old → at risk" |
| Baseline for ensemble models | Both | LR and DT are building blocks for Random Forest and Gradient Boosting |

> ➡️ **In Lesson 5 (the assignment lesson)**, you will apply both models end-to-end — making the choice between them part of the deliverable.


## Part 7: Getting Our Data

Let us import the required libraries:

- **`pandas`**, **`numpy`**: data manipulation and numerical computing
- **`matplotlib.pyplot`**: plotting the validation curve and confusion matrix
- **`sklearn.tree`**: `DecisionTreeClassifier` (the model) and `plot_tree` (visualization)
- **`sklearn.model_selection`**: `train_test_split` for the three-way split
- **`sklearn.pipeline`**: `Pipeline` to chain encoder → classifier
- **`sklearn.metrics`**: `accuracy_score`, `ConfusionMatrixDisplay` for evaluation
- **`category_encoders`**: `OrdinalEncoder` to convert categorical features to integers for the tree

The `category_encoders` library is a third-party package that extends scikit-learn's preprocessing. It handles unknown categories gracefully (assigning -1 by default) and integrates seamlessly with `Pipeline` — making it the standard choice for encoding in Decision Tree workflows.

**Code 4.3.0.1**: Import required libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from category_encoders import OrdinalEncoder
from sklearn.pipeline import Pipeline

# Set display options
# pd.set_option('display.max_columns', None)

Using the same `wrangle_nepal_data()` function from Lesson 1, load the Gorkha dataset. This gives us the same 70,836-building DataFrame we used in Lesson 2 — allowing a direct, apples-to-apples comparison between Logistic Regression (L2) and Decision Tree (L3).

> 📌 **Using the same data for comparison is intentional.** If we used different datasets for different models, we couldn't tell whether one model is genuinely better or just happened to see easier data.

**Code Task 4.3.1.1**: Load the Gorkha dataset using `wrangle_nepal_data`. Store in `df`.


In [ ]:
from duckdb_wrangle import wrangle_nepal_data

# Load Gorkha district data (district_id = 4) from the `data` folder
df = wrangle_nepal_data('./data', district_id=4)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nSevere damage rate: {df['severe_damage'].mean():.2%}")

---

## Part 8: Preparing the Data

We use the same feature set as Lesson 2 — all columns except `severe_damage`. This allows a direct comparison:

- **Same data**: same 70,836 Gorkha buildings
- **Same target**: `severe_damage` (binary: Grade 4/5 = 1, Grade 1-3 = 0)
- **Same features**: `age_building`, `height_ft_pre_eq`, `foundation_type`, `roof_type`, `ground_floor_type`, `other_floor_type`
- **Different model**: OrdinalEncoder + DecisionTreeClassifier (vs OHE + LogisticRegression in L2)

**Code Task 4.3.2.1**: Create `target` (= `'severe_damage'`) and `features` (all other column names). Create feature matrix `X` and target vector `y`.


In [ ]:
# Define target and features
target = 'severe_damage'
features = [col for col in df.columns if col != target]

# Create X and y
X = df[features]   # <-- feature matrix
y = df[target]   # <-- target vector

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Features: {features}")

---

## Part 9: Three-Way Split — Train/Validation/Test

In Lesson 2, we used a **two-way split** (train/test) and chose `max_depth=6` for the Decision Tree. But why `6` and not `5` or `8`? The answer requires a **third set**: the **validation set**.

### Why We Need Three Sets

| Set | Size | Purpose |
|-----|------|---------|
| **Training set** (60%) | ~42,500 buildings | Fit the model (find optimal splits) |
| **Validation set** (20%) | ~14,100 buildings | Tune hyperparameters (find best max_depth) |
| **Test set** (20%) | ~14,100 buildings | Final evaluation — only touched once |

**The problem with two-way splits for hyperparameter tuning:**

If we chose `max_depth` based on test set accuracy, we would be **fitting to the test set** — a form of data leakage. Every time we try a new `max_depth` and check the test accuracy, we're implicitly using test information to make modeling decisions.

The solution: use the validation set for hyperparameter tuning. The test set is truly untouched until the very end.

```
Full data (70,836 buildings)
├── Test set (20% = ~14,168)     ← Sealed until final evaluation
└── Remaining (80% = ~56,668)
    ├── Training set (75% of remaining = ~42,501)  ← Fit model
    └── Validation set (25% of remaining = ~14,167) ← Tune max_depth
```

> ⚠️ **Crucial rule:** the test set is only opened for the **final evaluation** after you have chosen all hyperparameters. Any earlier use of the test set contaminates your final evaluation.

**Code Task 4.3.3.1**: Split data into train (80%) and test (20%). Then split the training data into train (75%) and validation (25%). Use `random_state=42` for reproducibility.


In [ ]:
# First split: separate test set (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Second split: separate validation from training (25% of 80% = 20% overall)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X):.1%})")
print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X):.1%})")
print(f"Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X):.1%})")

✅ **You may now attempt Multiple Choice Question 4.3.3.1**

> 📊 **Verifying the split sizes:** with 70,836 total buildings:
> - Test: ~14,167 buildings (20%)
> - Train: ~42,501 buildings (60%)
> - Validation: ~14,168 buildings (20%)
> Total: 70,836 ✓

---

## Part 10: Baseline Model

Before training a Decision Tree, we establish the baseline: always predict the majority class.

**For Gorkha:** ~64% of buildings have `severe_damage = 1`. So the majority-class baseline achieves **~64% accuracy** by always predicting "severe."

Our Decision Tree must beat 64% to be useful. If it only achieves 65%, it has barely improved on a trivially naïve predictor.

**Code 4.3.4.1**: Calculate the baseline accuracy on the training set.


In [ ]:
# Calculate baseline (predict majority class)
baseline_acc = y_train.value_counts(normalize=True).max()

print(f"Baseline Accuracy: {baseline_acc:.4f}")
print(f"Majority class: {y_train.value_counts().idxmax()}")

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1169845945", h="3298dbabb7", width=700, height=450)

---

## Part 11: Building a Decision Tree Model

Decision trees use **OrdinalEncoder** (not OneHotEncoder) because:
1. Trees split on thresholds (`feature ≤ value`) — they do not compute weighted sums
2. OrdinalEncoder assigns integers (0, 1, 2...) which the tree uses as sortable values
3. The specific ordering of integers doesn't matter — the tree finds the optimal split threshold anyway
4. OHE would multiply the number of columns, slowing down split search

Our Pipeline: `OrdinalEncoder → DecisionTreeClassifier`

> 💡 **Starting with max_depth=6:** this is our initial guess. In Part 12, we systematically test different depths to find the optimal one using the validation set.

**Code Task 4.3.5.1**: Create a Pipeline with `OrdinalEncoder` and `DecisionTreeClassifier(max_depth=6, random_state=42)`. Fit it on the training data. Store in `dt_model`.


In [ ]:
# Create pipeline
model = Pipeline([
    ('encoder', OrdinalEncoder()),
    ('tree', DecisionTreeClassifier(max_depth=6, random_state=42))
])

# Fit the model
model.fit(X_train, y_train)

# Calculate accuracies
train_acc = accuracy_score(y_train, model.predict(X_train))
val_acc = accuracy_score(y_val, model.predict(X_val))

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Baseline Accuracy: {baseline_acc:.4f}")

✅ **You may now attempt Multiple Choice Question 4.3.5.1**

> 📊 **Initial depth=6 results:** note the training accuracy vs validation accuracy. Are they close (good generalization) or is there a large gap (overfitting)?

---

## Part 12: Hyperparameter Tuning — Finding the Best max_depth

With the validation set ready, we can systematically test different `max_depth` values and find the one that maximizes validation accuracy.

**The procedure:**
1. For each max_depth in [1, 2, 3, ..., 15]:
   - Train a new tree with that depth on the training set
   - Record training accuracy (on training set)
   - Record validation accuracy (on validation set)
2. Plot training accuracy vs validation accuracy across all depths
3. Pick the depth where validation accuracy peaks

> ⚠️ **We never use test accuracy for this comparison.** Only validation accuracy guides the hyperparameter choice. The test set stays sealed.

**Code Task 4.3.6.1**: Test max_depth values from 1 to 15. Store lists of training and validation accuracies in `train_accs` and `val_accs`.


In [ ]:
# Test different max_depth values
depths = range(1, 16)
train_accuracies = []
val_accuracies = []

for depth in depths:   # <-- "depth" is the loop variable
    # Create and train model
    model = Pipeline([
        ('encoder', OrdinalEncoder()),
        ('tree', DecisionTreeClassifier(max_depth=depth, random_state=42)) # <-- pass the loop variable here
    ])
    model.fit(X_train, y_train)

    # Calculate accuracies
    train_acc = accuracy_score(y_train, model.predict(X_train))
    val_acc = accuracy_score(y_val, model.predict(X_val))

    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

print("Hyperparameter tuning complete!")
print(f"Best validation accuracy: {max(val_accuracies):.4f} at depth {list(depths)[val_accuracies.index(max(val_accuracies))]}")

Now plot the validation curve to visualize the bias-variance tradeoff in action.

> 🔍 **What to look for in the plot:**
> - **Training accuracy** should increase monotonically as max_depth increases — deeper trees can always fit training data better
> - **Validation accuracy** should rise initially (reducing underfitting) and then plateau or decline (overfitting begins)
> - The **peak of the validation curve** is the optimal max_depth
> - The **gap between the two curves** at high depth is the visual signature of overfitting

**Code 4.3.6.1**: Plot the validation curve showing training and validation accuracy vs max_depth.


In [ ]:
# Find the depth with the best validation accuracy
best_depth = list(depths)[val_accuracies.index(max(val_accuracies))]

# Plot validation curve
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(depths, train_accuracies, 'o-', label='Training Accuracy', color='blue')
ax.plot(depths, val_accuracies, 'o-', label='Validation Accuracy', color='orange')
ax.axhline(y=baseline_acc, color='red', linestyle='--', label='Baseline')
ax.axvline(x=best_depth, color='green', linestyle='--', label=f'Best Depth = {best_depth}')
ax.set_xlabel('Max Depth')
ax.set_ylabel('Accuracy')
ax.set_title('Validation Curve: Decision Tree Max Depth')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

This plot reveals the **bias-variance tradeoff** in action:
- **Low max_depth (left side)**: Training accuracy ≈ validation accuracy ≈ 65% — both are low because the tree is too shallow to capture the patterns (underfitting / high bias)
- **Optimal max_depth**: Both curves peak. Validation accuracy is maximized.
- **High max_depth (right side)**: Training accuracy climbs toward 99% while validation accuracy falls back to ~68%. The growing gap shows overfitting (high variance).

> 📌 **The "crossing point"** where training and validation curves diverge is approximately where overfitting begins. Choose the depth just before this crossing — or at the validation peak.

✅ **You may now attempt Multiple Choice Question 4.3.6.1**

---

## Part 13: Training the Final Model

Based on the validation curve, select the `max_depth` where validation accuracy peaked. Train a fresh tree with this depth on the **combined train + validation data** (more data = better model), then evaluate on the test set.

> 💡 **Why combine train + validation for final training?** The validation set was only needed for hyperparameter selection. Once we know the best depth, there's no reason to withhold that data — training on more data generally improves performance.

**Code Task 4.3.7.1**: Find the `best_depth` (max_depth with highest validation accuracy). Train a new Pipeline with this depth. Store in `final_model`.


In [ ]:
# Train final model with best hyperparameter
final_model = Pipeline([
    ('encoder', OrdinalEncoder()),
    ('tree', DecisionTreeClassifier(max_depth=best_depth, random_state=42))
])
final_model.fit(X_train, y_train)

# Evaluate on test set
test_acc = accuracy_score(y_test, final_model.predict(X_test))

print(f"\nFinal Model Performance:")
print(f"Training Accuracy: {accuracy_score(y_train, final_model.predict(X_train)):.4f}")
print(f"Validation Accuracy: {accuracy_score(y_val, final_model.predict(X_val)):.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Baseline Accuracy: {baseline_acc:.4f}")

---

## Part 14: Feature Importance — Which Features Does the Tree Use?

Decision trees provide a natural measure of feature importance: **Gini reduction**. A feature is "important" if splits on that feature reduce Gini impurity a lot (separate the classes well) and are used frequently throughout the tree.

**How feature importance is computed:**
1. For each feature, sum up the Gini reduction for every split that used that feature
2. Weight each split by the number of samples that reached that node
3. Normalize so all importances sum to 1.0

**Interpreting the scores:**
- Importance = 0.40 for `foundation_type` → 40% of the total Gini reduction across all splits came from foundation type splits
- Higher importance = feature is more discriminative and used more often

> 🧠 **Decision Tree feature importance vs Logistic Regression odds ratios:**
> - **LR odds ratios** show *direction* (feature A multiplies odds by X) and magnitude
> - **DT feature importance** shows *magnitude only* — which features contribute most, but not whether they increase or decrease damage odds
> - They often agree on the most important features, but may rank them differently because they model different aspects of the relationship

**Code Task 4.3.8.1**: Extract feature importances from the fitted Pipeline. Sort them and create a horizontal bar chart.


In [ ]:
# Get feature importances
importances = final_model.named_steps['tree'].feature_importances_
feature_names = final_model.named_steps['encoder'].get_feature_names_out()

# Create Series and sort
feature_importance = pd.Series(importances, index=feature_names).sort_values(ascending=False)

print("Top 10 Most Important Features:")
print(feature_importance.head(10))

# Plot
fig, ax = plt.subplots(figsize=(9, 6))
feature_importance.head(10).plot(kind='barh', ax=ax)
ax.set_xlabel('Importance')
ax.set_title('Top 10 Feature Importances (Decision Tree)')
plt.tight_layout()
plt.show()

✅ **You may now attempt Multiple Choice Question 4.3.8.1**

> 📊 **Reading the feature importance chart:** the most important features should align with physical intuition about earthquake damage:
> - `foundation_type`: the primary structural material bearing the earthquake load — typically ranks #1 or #2
> - `roof_type`: heavy roofs (stone/tile) collapse more easily than light roofs (zinc/tin) — often ranks highly
> - `age_building`: older construction predates modern earthquake codes — tends to be moderately important
> - `height_ft_pre_eq`: taller buildings have more momentum during shaking — some importance
>
> If the top features match seismic engineering knowledge, the tree has learned genuine physical patterns, not just statistical noise.


In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1169845836", h="3298dbabb7", width=700, height=450)

---

## Part 15: Visualizing the Tree Structure

One of Decision Tree's greatest strengths is **interpretability**: you can visualize the tree and trace exactly how any prediction is made.

We limit visualization to the first 2 levels (`max_depth=2` in `plot_tree`) for readability — the full tree is typically too large to display.

> 🔍 **What to look for in the tree diagram:**
> - The **root node** (top): the single most important split — the feature that reduces Gini impurity the most overall
> - **Node colors**: orange = predominantly "not severe"; blue = predominantly "severe"; darker = purer
> - **Gini values**: smaller = purer node; the root node Gini should be close to `2 × 0.64 × 0.36 ≈ 0.46` (our class distribution)
> - **The split criterion**: `feature ≤ X.X` — trace the path for any specific building

**Code 4.3.9.1**: Visualize the decision tree (first 2 levels for readability).


In [ ]:
# Visualize the tree (first 2 levels)
fig, ax = plt.subplots(figsize=(9, 6))
plot_tree(
    final_model.named_steps['tree'],
    feature_names=feature_names,
    class_names=['Not Severe', 'Severe'],
    filled=True,
    rounded=True,
    max_depth=2,
    ax=ax,
    fontsize=10
)
ax.set_title('Decision Tree Structure (First 2 Levels)')
plt.show()

### Understanding the Decision Tree Visualization

Each colored box in the diagram above is a **node** — a decision point or a final prediction.

**What each element in a node means:**

| Element | What it shows | Example |
|---------|--------------|---------|
| `foundation_type <= 2.5` | Split criterion: feature ≤ threshold | Ordinal codes 0, 1, 2 go left; 3, 4 go right |
| `gini = 0.46` | Node impurity: how mixed are the classes? | 0 = pure, 0.5 = completely mixed |
| `samples = 42501` | Number of training buildings at this node | Root: all training buildings |
| `value = [15300, 27201]` | Counts of [not severe, severe] at this node | 15,300 not severe, 27,201 severe |
| **Color (blue)** | Dominant class is severe (1) | Darker blue = purer "severe" node |
| **Color (orange)** | Dominant class is not severe (0) | Darker orange = purer "not severe" node |

**Tracing a prediction path:**

To predict a specific building:
1. Start at the root node
2. At each node: if the building's feature value ≤ the threshold, go LEFT; if > threshold, go RIGHT
3. Continue until you reach a leaf node
4. The leaf node's dominant class is the prediction

**Example trace for a building with `foundation_type = 3` (ordinal-encoded strong foundation):**
```
Root: foundation_type <= 2.5?  → NO (3 > 2.5) → Go RIGHT
Right node: ...ask next question...
→ Continue until leaf
→ Final prediction at leaf
```

> 💡 **Why do tree visualizations use ordinal-encoded values?** The tree sees the `OrdinalEncoder` output (integers 0–4), not the original strings. To interpret `foundation_type <= 2.5`, you need to know which original categories map to codes 0, 1, 2 vs. 3, 4.


In [ ]:
# Note: From Lesson 2, Logistic Regression had ~70.6% test accuracy
logistic_regression_accuracy = 0.706  # From Lesson 2

print("Model Comparison:")
print("=" * 50)
print(f"{'Model':<25} {'Test Accuracy':<15}")
print("=" * 50)
print(f"{'Baseline':<25} {baseline_acc:<15.4f}")
print(f"{'Logistic Regression':<25} {logistic_regression_accuracy:<15.4f}")
print(f"{'Decision Tree':<25} {test_acc:<15.4f}")
print("=" * 50)

# Calculate improvements
print(f"\nImprovement over baseline:")
print(f"  Logistic Regression: +{logistic_regression_accuracy - baseline_acc:.4f}")
print(f"  Decision Tree: +{test_acc - baseline_acc:.4f}")

if test_acc > logistic_regression_accuracy:
    print(f"\n✓ Decision Tree wins by {test_acc - logistic_regression_accuracy:.4f}")
else:
    print(f"\n✗ Logistic Regression wins by {logistic_regression_accuracy - test_acc:.4f}")

✅ **You may now attempt Multiple Choice Question 4.3.10.1**

> 📊 **Comparing Decision Tree to Logistic Regression:**
> - Both models achieve similar test accuracy (~71-72%)
> - This suggests the problem is approximately linearly separable with the features we have
> - If the Decision Tree significantly outperforms LR (e.g., 75% vs 71%), it suggests important non-linear interactions
> - If they perform similarly, LR's interpretability (odds ratios) may be preferred for stakeholder communication

---

## Summary and Discussion

This lesson built a complete Decision Tree pipeline from scratch, including proper three-way split, hyperparameter tuning via validation curve, and feature importance interpretation.

### What You Built and Learned

| Concept | Key takeaway |
|---------|-------------|
| **Recursive splitting** | Trees make predictions by asking sequential yes/no questions; each split finds the best Gini-reducing threshold |
| **Axis-aligned boundaries** | Trees create rectangular regions in feature space; cannot create diagonal boundaries without many splits |
| **Gini impurity** | Measures node purity: 0 = pure, 0.5 = maximally mixed; trees minimize Gini at each split |
| **OrdinalEncoder** | Trees use ordinal integers for split thresholds; OHE is unnecessary and adds columns |
| **Three-way split** | Train (60%) for fitting, validation (20%) for hyperparameter tuning, test (20%) for final evaluation |
| **Validation curve** | Plots training vs validation accuracy across max_depth; peak validation = optimal depth |
| **Bias-variance tradeoff** | Shallow tree = high bias (underfitting); deep tree = high variance (overfitting) |
| **Feature importance** | Gini reduction summed across all splits; top features are most discriminative |
| **plot_tree** | Visualize the tree to trace any prediction path from root to leaf |

### Key Insights

- Decision Tree achieves **~71% accuracy**, similar to Logistic Regression's ~71% — both models see the same patterns
- `foundation_type` and `roof_type` are consistently the most important features in both LR (by odds ratio) and DT (by Gini reduction)
- The validation curve confirmed the optimal max_depth (around 6-8 for Nepal data) and showed the bias-variance tradeoff visually
- The tree visualization reveals that the **root split** (the single most important question) is always about `foundation_type` — physical engineering knowledge aligns with the model's learned hierarchy

### Decision Tree vs Logistic Regression: When to Use Each

| Scenario | Preferred model | Why |
|----------|----------------|-----|
| Non-linear feature interactions | Decision Tree | Rectangular splits capture interactions naturally |
| Interpretability for stakeholders | Decision Tree | Can show visual path: "This building was classified because..." |
| Probability calibration needed | Logistic Regression | LR outputs calibrated probabilities; DT leaf proportions are less reliable |
| Many features with some irrelevant | Decision Tree | Tree naturally ignores low-importance features |
| Linear additive effects | Logistic Regression | Cleaner model when effects are truly additive |
| Need for regularization theory | Logistic Regression | L1/L2 regularization has cleaner theoretical grounding |

### Discussion Questions

1. Why does the Decision Tree use `OrdinalEncoder` while Logistic Regression uses `OneHotEncoder`? What property of trees makes ordinal encoding acceptable?
2. Looking at the validation curve, at which max_depth does overfitting begin? What visual cue tells you this?
3. Both Decision Tree and Logistic Regression achieved similar accuracy on Nepal data. What does this similarity suggest about the nature of the damage prediction problem?
4. Feature importance shows Gini reduction but no direction. How would you find the *direction* of a Decision Tree feature's effect (does more of it increase or decrease damage probability)?
5. If you wanted to maximize recall (catching all severely damaged buildings), would you prefer to tune max_depth or adjust the decision threshold? What are the trade-offs?
6. A stakeholder says "the Decision Tree is transparent because we can see the tree." But the full tree has thousands of nodes. Is that really transparent? How would you address this objection?

### Next Steps

In Lesson 4, you will:
- Join the demographic data (`household_demographics`) to the building data using SQL
- Analyze **caste-based differences** in severe damage rates — an equity analysis
- Understand whether some communities were disproportionately affected by the earthquake
- Add demographic features to the model and assess whether they improve predictions

> ➡️ Lesson 4 returns to the SQL skills from Lesson 1 — you will write a multi-table JOIN to add the `caste_household` and `gender_household_head` columns to the feature matrix.
